In [2]:
import pandas as pd
import numpy as np
from Parser import Parser
from processing_types import (
    NormalizationStrategy, EncodingStrategy,
    MissingValuesNumericStrategy, MissingValuesCategoricalStrategy, RetentionPolicy
)
from sklearn.metrics import classification_report
from sklearn.svm import SVC
import time
import json
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, precision_score, recall_score, f1_score
from pathlib import Path
from sklearn.model_selection import GridSearchCV
from statistics import mean, stdev
import os


In [13]:
NUM_SPLITS = 10
dataset_name = ""
output_dir = "./results"
os.makedirs(output_dir, exist_ok=True)
parser = Parser(base_path="../datasetsCBR/datasetsCBR", dataset_name= dataset_name,
                  normalization_strategy=NormalizationStrategy.STANDARDIZE,
                  encoding_strategy=EncodingStrategy.LABEL_ENCODE,
                  missing_values_numeric_strategy=MissingValuesNumericStrategy.DROP,
                  missing_values_categorical_strategy=MissingValuesCategoricalStrategy.MODE)



In [14]:
kernels = ['rbf', 'poly']
rows = []

for kernel in kernels:
    print(f"  Training kernel = {kernel}")
    for split_idx in range(NUM_SPLITS):
        print(f"\n=== Split {split_idx+1}/{NUM_SPLITS} ===")

        #  Load split
        train_matrix, test_matrix = parser.get_split(split_idx)
        np_train_matrix = train_matrix.reset_index(drop=True).to_numpy()
        np_test_matrix = test_matrix.reset_index(drop=True).to_numpy()

        X_train, y_train = np_train_matrix[:, :-1], np_train_matrix[:, -1]
        X_test, y_test = np_test_matrix[:, :-1], np_test_matrix[:, -1]

        n_train = len(X_train)
        n_test = len(X_test)

        #  Train
        start_fit = time.time()
        clf = SVC(kernel=kernel)
        clf.fit(X_train, y_train)
        fit_time = time.time() - start_fit

        #  Predict
        start_pred = time.time()
        y_pred = clf.predict(X_test)
        pred_time = time.time() - start_pred
        total_time = fit_time + pred_time

        #  Metrics
        acc = accuracy_score(y_test, y_pred)
        pM = precision_score(y_test, y_pred, average='macro', zero_division=0)
        rM = recall_score(y_test, y_pred, average='macro', zero_division=0)
        fM = f1_score(y_test, y_pred, average='macro', zero_division=0)

        pW = precision_score(y_test, y_pred, average='weighted', zero_division=0)
        rW = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        fW = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        labels = np.unique(np.concatenate((y_test, y_pred)))
        cm = confusion_matrix(y_test, y_pred, labels=labels)

        #  Append row
        rows.append({
            "dataset": dataset_name,
            "split": split_idx,
            "kernel": kernel,

            "n_train": n_train,
            "n_test": n_test,

            "fit_time_s": fit_time,
            "predict_time_s": pred_time,
            "total_time_s": total_time,

            "accuracy": acc,
            "precision_macro": pM,
            "recall_macro": rM,
            "f1_macro": fM,

            "precision_weighted": pW,
            "recall_weighted": rW,
            "f1_weighted": fW,

            "confusion_matrix_json": cm.tolist(),
    })

#  Convert to DataFrame
df = pd.DataFrame(rows)

#  Compute summary per kernel
summary = (
    df.groupby("kernel")
    .agg({
        "accuracy": ["mean", "std"],
        "precision_macro": ["mean", "std"],
        "recall_macro": ["mean", "std"],
        "f1_macro": ["mean", "std"],
        "precision_weighted": ["mean", "std"],
        "recall_weighted": ["mean", "std"],
        "f1_weighted": ["mean", "std"],
        "fit_time_s": ["mean", "std"],
        "predict_time_s": ["mean", "std"],
        "total_time_s": ["mean", "std"],
    })
)
summary.columns = ["_".join(col) for col in summary.columns]
summary.reset_index(inplace=True)

# Save results
folds_path = os.path.join(output_dir, f"svm_{dataset_name}_folds_results.csv")
summary_path = os.path.join(output_dir, f"svm_{dataset_name}_summary.csv")

df.to_csv(folds_path, index=False)
summary.to_csv(summary_path, index=False)

  Training kernel = rbf

=== Split 1/10 ===

=== Split 2/10 ===

=== Split 3/10 ===

=== Split 4/10 ===

=== Split 5/10 ===

=== Split 6/10 ===

=== Split 7/10 ===

=== Split 8/10 ===

=== Split 9/10 ===

=== Split 10/10 ===
  Training kernel = poly

=== Split 1/10 ===

=== Split 2/10 ===

=== Split 3/10 ===

=== Split 4/10 ===

=== Split 5/10 ===

=== Split 6/10 ===

=== Split 7/10 ===

=== Split 8/10 ===

=== Split 9/10 ===

=== Split 10/10 ===
